# ChemAI: Predict the Cure

Предсказание IC50, CC50 и SI для химических соединений против вируса гриппа.

**Подход:**
1. log-трансформация таргетов (значения строго > 0, логнормальное распределение)
2. Ансамбль LightGBM + XGBoost + CatBoost, 5-fold CV
3. Optuna для подбора гиперпараметров LGB и XGB
4. IC50 и CC50 обучаются первыми; их OOF-предсказания используются как признаки для SI (стекинг)
5. SI: прямое предсказание + формула CC50/IC50, бленд по OOF-качеству
6. Взвешенный ансамбль (вес ∝ 1/OOF-RMSE)

## 1. Импорты и константы

In [1]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
from sklearn.model_selection import KFold
from sklearn.impute import SimpleImputer
from sklearn.metrics import mean_squared_error
import lightgbm as lgb
import xgboost as xgb
from catboost import CatBoostRegressor
import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)

# ─── Константы ───────────────────────────────────────────────
SEED              = 42
N_FOLDS           = 5
OPTUNA_TRIALS_LGB = 50
OPTUNA_TRIALS_XGB = 40
USE_OPTUNA        = True
CORR_THRESHOLD    = 0.95

TARGET_IC50 = 'IC50, mM'
TARGET_CC50 = 'CC50, mM'
TARGET_SI   = 'SI'
TARGETS     = [TARGET_IC50, TARGET_CC50, TARGET_SI]

# Известные положительные дескрипторы для log-дополнения
POSITIVE_FEATS = [
    'MolWt', 'HeavyAtomMolWt', 'ExactMolWt', 'TPSA',
    'LabuteASA', 'BertzCT', 'Ipc',
    'Chi0', 'Chi0n', 'Chi0v', 'MolMR',
]

np.random.seed(SEED)
print('Константы загружены. SEED =', SEED)

Константы загружены. SEED = 42


## 2. Загрузка данных

In [2]:
train = pd.read_csv('../data/train.csv')
test  = pd.read_csv('../data/test.csv')

print(f'Train: {train.shape}, Test: {test.shape}')

# Проверяем инвариант SI = CC50 / IC50
si_calc = train[TARGET_CC50] / train[TARGET_IC50]
match   = (np.abs(train[TARGET_SI] - si_calc) < 1e-6).mean()
print(f'SI = CC50/IC50: совпадение {match:.1%}')

print('\nТаргеты (описательная статистика):')
print(train[TARGETS].describe().round(2))

Train: (751, 214), Test: (250, 211)
SI = CC50/IC50: совпадение 100.0%

Таргеты (описательная статистика):
       IC50, mM  CC50, mM        SI
count    751.00    751.00    751.00
mean     204.54    577.43     89.15
std      370.37    641.52    788.88
min        0.00      0.70      0.01
25%       13.22    100.00      1.50
50%       44.07    376.58      4.00
75%      206.79    877.51     17.37
max     4095.19   4538.98  15620.60


## 3. Препроцессинг признаков

In [3]:
feat_cols = [c for c in train.columns if c not in ['index'] + TARGETS]
print(f'Признаков исходно: {len(feat_cols)}')

# Удаляем константные признаки
const_feats = [c for c in feat_cols if train[c].nunique() <= 1]
feat_cols   = [c for c in feat_cols if c not in const_feats]
print(f'Удалено константных: {len(const_feats)}')

# Заполняем пропуски медианой (fit только на train)
imputer = SimpleImputer(strategy='median')
X_train = pd.DataFrame(imputer.fit_transform(train[feat_cols]), columns=feat_cols)
X_test  = pd.DataFrame(imputer.transform(test[feat_cols]),      columns=feat_cols)

# Добавляем log-версии положительных дескрипторов
for feat in POSITIVE_FEATS:
    if feat in feat_cols and (X_train[feat] > 0).all():
        X_train[f'log_{feat}'] = np.log(X_train[feat])
        X_test[f'log_{feat}']  = np.log(X_test[feat])
print(f'После log-дополнения: {X_train.shape[1]}')

# Удаляем высококоррелированные признаки (порог > 0.95)
corr_m    = X_train.corr().abs()
upper     = corr_m.where(np.triu(np.ones(corr_m.shape), k=1).astype(bool))
drop_corr = [c for c in upper.columns if any(upper[c] > CORR_THRESHOLD)]
X_train   = X_train.drop(columns=drop_corr)
X_test    = X_test.drop(columns=drop_corr)
print(f'Удалено коррелированных (r>{CORR_THRESHOLD}): {len(drop_corr)}')
print(f'Итоговых признаков: {X_train.shape[1]}')

Признаков исходно: 210
Удалено константных: 18
После log-дополнения: 202


Удалено коррелированных (r>0.95): 42
Итоговых признаков: 160


## 4. Логарифмирование таргетов

In [4]:
y_ic50_log = np.log(train[TARGET_IC50].values)
y_cc50_log = np.log(train[TARGET_CC50].values)
y_si_log   = np.log(train[TARGET_SI].values)

print('Таргеты логарифмированы (np.log).')
for name, y in [('IC50', y_ic50_log), ('CC50', y_cc50_log), ('SI', y_si_log)]:
    print(f'  log({name}): mean={y.mean():.2f}, std={y.std():.2f}, '
          f'min={y.min():.2f}, max={y.max():.2f}')

Таргеты логарифмированы (np.log).
  log(IC50): mean=3.78, std=2.15, min=-5.65, max=8.32
  log(CC50): mean=5.52, std=1.64, min=-0.36, max=8.42
  log(SI): mean=1.74, std=1.76, min=-4.47, max=9.66


## 5. Функции моделей и Optuna

In [5]:
# ─── Параметры по умолчанию ──────────────────────────────────

DEFAULT_LGB = {
    'objective': 'regression', 'metric': 'rmse',
    'n_estimators': 3000, 'learning_rate': 0.02,
    'num_leaves': 20, 'max_depth': 5,
    'min_child_samples': 20, 'subsample': 0.7, 'subsample_freq': 1,
    'colsample_bytree': 0.7, 'reg_alpha': 1.0, 'reg_lambda': 5.0,
    'random_state': SEED, 'n_jobs': -1, 'verbose': -1,
}

DEFAULT_XGB = {
    'objective': 'reg:squarederror', 'n_estimators': 3000,
    'learning_rate': 0.02, 'max_depth': 4, 'min_child_weight': 10,
    'subsample': 0.7, 'colsample_bytree': 0.7,
    'reg_alpha': 1.0, 'reg_lambda': 5.0,
    'early_stopping_rounds': 150,
    'random_state': SEED, 'n_jobs': -1, 'verbosity': 0,
}

DEFAULT_CAT = {
    'iterations': 3000, 'learning_rate': 0.02, 'depth': 6,
    'l2_leaf_reg': 3.0, 'subsample': 0.8, 'colsample_bylevel': 0.7,
    'random_seed': SEED, 'verbose': 0,
}


# ─── Optuna: LightGBM ────────────────────────────────────────

def _lgb_objective(trial, X, y):
    params = {
        'objective': 'regression', 'metric': 'rmse', 'n_estimators': 2000,
        'learning_rate':     trial.suggest_float('learning_rate', 0.01, 0.1, log=True),
        'num_leaves':        trial.suggest_int('num_leaves', 8, 63),
        'max_depth':         trial.suggest_int('max_depth', 3, 8),
        'min_child_samples': trial.suggest_int('min_child_samples', 10, 60),
        'subsample':         trial.suggest_float('subsample', 0.5, 1.0),
        'subsample_freq':    1,
        'colsample_bytree':  trial.suggest_float('colsample_bytree', 0.4, 1.0),
        'reg_alpha':         trial.suggest_float('reg_alpha', 0.01, 20.0, log=True),
        'reg_lambda':        trial.suggest_float('reg_lambda', 0.01, 20.0, log=True),
        'min_split_gain':    trial.suggest_float('min_split_gain', 0.0, 1.0),
        'random_state': SEED, 'n_jobs': -1, 'verbose': -1,
    }
    kf = KFold(n_splits=3, shuffle=True, random_state=SEED)
    rmses = []
    for tr_idx, val_idx in kf.split(X):
        m = lgb.LGBMRegressor(**params)
        m.fit(X.iloc[tr_idx], y[tr_idx],
              eval_set=[(X.iloc[val_idx], y[val_idx])],
              callbacks=[lgb.early_stopping(80, verbose=False), lgb.log_evaluation(-1)])
        rmses.append(np.sqrt(mean_squared_error(y[val_idx], m.predict(X.iloc[val_idx]))))
    return float(np.mean(rmses))


def tune_lgb(X, y, target_name):
    print(f'  Optuna LGB [{target_name}]: {OPTUNA_TRIALS_LGB} trials (3-fold)...')
    study = optuna.create_study(direction='minimize',
                                sampler=optuna.samplers.TPESampler(seed=SEED))
    study.optimize(lambda t: _lgb_objective(t, X, y),
                   n_trials=OPTUNA_TRIALS_LGB, show_progress_bar=False)
    best = study.best_params
    best.update({'objective': 'regression', 'metric': 'rmse', 'n_estimators': 3000,
                 'subsample_freq': 1, 'random_state': SEED, 'n_jobs': -1, 'verbose': -1})
    print(f'  CV RMSE LGB: {study.best_value:.4f} | '
          f'lr={best["learning_rate"]:.4f} leaves={best["num_leaves"]} depth={best["max_depth"]}')
    return best


# ─── Optuna: XGBoost ─────────────────────────────────────────

def _xgb_objective(trial, X, y):
    params = {
        'objective': 'reg:squarederror', 'n_estimators': 2000,
        'learning_rate':    trial.suggest_float('learning_rate', 0.01, 0.1, log=True),
        'max_depth':        trial.suggest_int('max_depth', 3, 8),
        'min_child_weight': trial.suggest_int('min_child_weight', 3, 40),
        'subsample':        trial.suggest_float('subsample', 0.5, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.4, 1.0),
        'reg_alpha':        trial.suggest_float('reg_alpha', 0.01, 20.0, log=True),
        'reg_lambda':       trial.suggest_float('reg_lambda', 0.01, 20.0, log=True),
        'early_stopping_rounds': 80,
        'random_state': SEED, 'n_jobs': -1, 'verbosity': 0,
    }
    kf = KFold(n_splits=3, shuffle=True, random_state=SEED)
    rmses = []
    for tr_idx, val_idx in kf.split(X):
        m = xgb.XGBRegressor(**params)
        m.fit(X.iloc[tr_idx], y[tr_idx],
              eval_set=[(X.iloc[val_idx], y[val_idx])], verbose=False)
        rmses.append(np.sqrt(mean_squared_error(y[val_idx], m.predict(X.iloc[val_idx]))))
    return float(np.mean(rmses))


def tune_xgb(X, y, target_name):
    print(f'  Optuna XGB [{target_name}]: {OPTUNA_TRIALS_XGB} trials (3-fold)...')
    study = optuna.create_study(direction='minimize',
                                sampler=optuna.samplers.TPESampler(seed=SEED))
    study.optimize(lambda t: _xgb_objective(t, X, y),
                   n_trials=OPTUNA_TRIALS_XGB, show_progress_bar=False)
    best = study.best_params
    best.update({'objective': 'reg:squarederror', 'n_estimators': 3000,
                 'early_stopping_rounds': 150,
                 'random_state': SEED, 'n_jobs': -1, 'verbosity': 0})
    print(f'  CV RMSE XGB: {study.best_value:.4f} | '
          f'lr={best["learning_rate"]:.4f} depth={best["max_depth"]} mcw={best["min_child_weight"]}')
    return best


# ─── Кросс-валидация: 3 модели ───────────────────────────────

def cross_val_predict_all(X_tr, y_log, X_te, lgb_params, xgb_params, cat_params):
    """5-fold CV, возвращает OOF и test предсказания в log-пространстве."""
    kf = KFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)
    n_tr, n_te = len(X_tr), len(X_te)

    oof_lgb = np.zeros(n_tr);  pred_lgb = np.zeros(n_te)
    oof_xgb = np.zeros(n_tr);  pred_xgb = np.zeros(n_te)
    oof_cat = np.zeros(n_tr);  pred_cat = np.zeros(n_te)

    for fold, (tr_idx, val_idx) in enumerate(kf.split(X_tr)):
        Xf_tr, Xf_val = X_tr.iloc[tr_idx], X_tr.iloc[val_idx]
        yf_tr, yf_val = y_log[tr_idx],      y_log[val_idx]

        m_lgb = lgb.LGBMRegressor(**lgb_params)
        m_lgb.fit(Xf_tr, yf_tr, eval_set=[(Xf_val, yf_val)],
                  callbacks=[lgb.early_stopping(150, verbose=False), lgb.log_evaluation(-1)])
        oof_lgb[val_idx] = m_lgb.predict(Xf_val)
        pred_lgb        += m_lgb.predict(X_te) / N_FOLDS

        m_xgb = xgb.XGBRegressor(**xgb_params)
        m_xgb.fit(Xf_tr, yf_tr, eval_set=[(Xf_val, yf_val)], verbose=False)
        oof_xgb[val_idx] = m_xgb.predict(Xf_val)
        pred_xgb        += m_xgb.predict(X_te) / N_FOLDS

        m_cat = CatBoostRegressor(**cat_params)
        m_cat.fit(Xf_tr, yf_tr, eval_set=(Xf_val, yf_val),
                  early_stopping_rounds=150, verbose=0)
        oof_cat[val_idx] = m_cat.predict(Xf_val)
        pred_cat        += m_cat.predict(X_te) / N_FOLDS

        r = {k: np.sqrt(mean_squared_error(yf_val, v))
             for k, v in [('LGB', oof_lgb[val_idx]),
                           ('XGB', oof_xgb[val_idx]),
                           ('CAT', oof_cat[val_idx])]}
        print(f'  Fold {fold+1}: LGB={r["LGB"]:.4f} XGB={r["XGB"]:.4f} CAT={r["CAT"]:.4f}')

    r_lgb = np.sqrt(mean_squared_error(y_log, oof_lgb))
    r_xgb = np.sqrt(mean_squared_error(y_log, oof_xgb))
    r_cat = np.sqrt(mean_squared_error(y_log, oof_cat))
    print(f'  OOF: LGB={r_lgb:.4f} XGB={r_xgb:.4f} CAT={r_cat:.4f}')

    w = np.array([1/r_lgb, 1/r_xgb, 1/r_cat])
    w /= w.sum()
    print(f'  Веса: LGB={w[0]:.3f} XGB={w[1]:.3f} CAT={w[2]:.3f}')

    oof_ens  = w[0]*oof_lgb  + w[1]*oof_xgb  + w[2]*oof_cat
    pred_ens = w[0]*pred_lgb + w[1]*pred_xgb + w[2]*pred_cat
    print(f'  Ensemble OOF: {np.sqrt(mean_squared_error(y_log, oof_ens)):.4f}')

    return oof_ens, pred_ens


print('Функции определены.')

Функции определены.


## 6. Обучение: IC50

In [6]:
print('=' * 50)
print('TARGET: IC50')
print('=' * 50)

if USE_OPTUNA:
    lgb_ic50 = tune_lgb(X_train, y_ic50_log, 'IC50')
    xgb_ic50 = tune_xgb(X_train, y_ic50_log, 'IC50')
else:
    lgb_ic50 = DEFAULT_LGB.copy()
    xgb_ic50 = DEFAULT_XGB.copy()

oof_ic50_log, pred_ic50_log = cross_val_predict_all(
    X_train, y_ic50_log, X_test,
    lgb_ic50, xgb_ic50, DEFAULT_CAT
)

TARGET: IC50
  Optuna LGB [IC50]: 50 trials (3-fold)...


  CV RMSE LGB: 1.5922 | lr=0.0410 leaves=39 depth=4
  Optuna XGB [IC50]: 40 trials (3-fold)...


  CV RMSE XGB: 1.5891 | lr=0.0800 depth=7 mcw=19


  Fold 1: LGB=1.5175 XGB=1.5894 CAT=1.5519


  Fold 2: LGB=1.4661 XGB=1.4442 CAT=1.4542


  Fold 3: LGB=1.6912 XGB=1.6688 CAT=1.7089


  Fold 4: LGB=1.6129 XGB=1.6806 CAT=1.6387


  Fold 5: LGB=1.6159 XGB=1.6305 CAT=1.5893
  OOF: LGB=1.5826 XGB=1.6050 CAT=1.5909
  Веса: LGB=0.335 XGB=0.331 CAT=0.334
  Ensemble OOF: 1.5829


## 7. Обучение: CC50

In [7]:
print('=' * 50)
print('TARGET: CC50')
print('=' * 50)

if USE_OPTUNA:
    lgb_cc50 = tune_lgb(X_train, y_cc50_log, 'CC50')
    xgb_cc50 = tune_xgb(X_train, y_cc50_log, 'CC50')
else:
    lgb_cc50 = DEFAULT_LGB.copy()
    xgb_cc50 = DEFAULT_XGB.copy()

oof_cc50_log, pred_cc50_log = cross_val_predict_all(
    X_train, y_cc50_log, X_test,
    lgb_cc50, xgb_cc50, DEFAULT_CAT
)

TARGET: CC50
  Optuna LGB [CC50]: 50 trials (3-fold)...


  CV RMSE LGB: 1.2866 | lr=0.0207 leaves=29 depth=5
  Optuna XGB [CC50]: 40 trials (3-fold)...


  CV RMSE XGB: 1.2892 | lr=0.0112 depth=4 mcw=14


  Fold 1: LGB=1.1770 XGB=1.1781 CAT=1.1776


  Fold 2: LGB=1.2152 XGB=1.2135 CAT=1.2026


  Fold 3: LGB=1.4224 XGB=1.4114 CAT=1.4086


  Fold 4: LGB=1.3272 XGB=1.3224 CAT=1.3201


  Fold 5: LGB=1.1869 XGB=1.2042 CAT=1.2139
  OOF: LGB=1.2692 XGB=1.2689 CAT=1.2674
  Веса: LGB=0.333 XGB=0.333 CAT=0.334
  Ensemble OOF: 1.2624


## 8. Обучение: SI (стекинг + формула)

In [8]:
print('=' * 50)
print('TARGET: SI (+ OOF IC50/CC50 как признаки)')
print('=' * 50)

# Стекинг: OOF-предсказания IC50/CC50 как дополнительные признаки
X_train_si = X_train.copy()
X_train_si['oof_log_ic50']       = oof_ic50_log
X_train_si['oof_log_cc50']       = oof_cc50_log
X_train_si['oof_log_si_formula'] = oof_cc50_log - oof_ic50_log

X_test_si = X_test.copy()
X_test_si['oof_log_ic50']        = pred_ic50_log
X_test_si['oof_log_cc50']        = pred_cc50_log
X_test_si['oof_log_si_formula']  = pred_cc50_log - pred_ic50_log

if USE_OPTUNA:
    lgb_si = tune_lgb(X_train_si, y_si_log, 'SI')
    xgb_si = tune_xgb(X_train_si, y_si_log, 'SI')
else:
    lgb_si = DEFAULT_LGB.copy()
    xgb_si = DEFAULT_XGB.copy()

oof_si_direct_log, pred_si_direct_log = cross_val_predict_all(
    X_train_si, y_si_log, X_test_si,
    lgb_si, xgb_si, DEFAULT_CAT
)

TARGET: SI (+ OOF IC50/CC50 как признаки)
  Optuna LGB [SI]: 50 trials (3-fold)...


  CV RMSE LGB: 1.5206 | lr=0.0753 leaves=36 depth=6
  Optuna XGB [SI]: 40 trials (3-fold)...


  CV RMSE XGB: 1.5099 | lr=0.0410 depth=7 mcw=9


  Fold 1: LGB=1.4389 XGB=1.4013 CAT=1.4060


  Fold 2: LGB=1.3960 XGB=1.3255 CAT=1.4451


  Fold 3: LGB=1.6889 XGB=1.6752 CAT=1.6865


  Fold 4: LGB=1.4941 XGB=1.4874 CAT=1.4755


  Fold 5: LGB=1.4297 XGB=1.4153 CAT=1.4349
  OOF: LGB=1.4931 XGB=1.4657 CAT=1.4929
  Веса: LGB=0.331 XGB=0.337 CAT=0.331
  Ensemble OOF: 1.4678


## 9. Бленд SI: прямое предсказание + формула

In [9]:
# Формула: log(SI) = log(CC50) - log(IC50)
oof_si_formula_log  = oof_cc50_log  - oof_ic50_log
pred_si_formula_log = pred_cc50_log - pred_ic50_log

rmse_direct  = np.sqrt(mean_squared_error(y_si_log, oof_si_direct_log))
rmse_formula = np.sqrt(mean_squared_error(y_si_log, oof_si_formula_log))
print(f'SI OOF RMSE (log): прямое+стек={rmse_direct:.4f}  формула={rmse_formula:.4f}')

# Вес обратно пропорционален RMSE
alpha_direct  = rmse_formula / (rmse_direct + rmse_formula)
alpha_formula = 1.0 - alpha_direct
print(f'Вес прямого: {alpha_direct:.3f}  вес формулы: {alpha_formula:.3f}')

oof_si_log  = alpha_direct * oof_si_direct_log  + alpha_formula * oof_si_formula_log
pred_si_log = alpha_direct * pred_si_direct_log + alpha_formula * pred_si_formula_log

SI OOF RMSE (log): прямое+стек=1.4678  формула=1.5086
Вес прямого: 0.507  вес формулы: 0.493


## 10. Оценка качества (OOF RMSE)

In [10]:
pred_ic50 = np.exp(pred_ic50_log)
pred_cc50 = np.exp(pred_cc50_log)
pred_si   = np.exp(pred_si_log)

rmse_ic50 = np.sqrt(mean_squared_error(train[TARGET_IC50], np.exp(oof_ic50_log)))
rmse_cc50 = np.sqrt(mean_squared_error(train[TARGET_CC50], np.exp(oof_cc50_log)))
rmse_si   = np.sqrt(mean_squared_error(train[TARGET_SI],   np.exp(oof_si_log)))
rmse_avg  = (rmse_ic50 + rmse_cc50 + rmse_si) / 3

print('=' * 50)
print('CV RMSE в исходном пространстве (OOF):')
print(f'  IC50   : {rmse_ic50:.4f}')
print(f'  CC50   : {rmse_cc50:.4f}')
print(f'  SI     : {rmse_si:.4f}')
print(f'  Среднее: {rmse_avg:.4f}')
print('=' * 50)

# Калибровка: медианы должны быть близки к тренировочным
print('\nКалибровка (сравнение медиан):')
for col, pred, name in [
    (TARGET_IC50, pred_ic50, 'IC50'),
    (TARGET_CC50, pred_cc50, 'CC50'),
    (TARGET_SI,   pred_si,   'SI'),
]:
    tm = train[col].median()
    pm = np.median(pred)
    flag = '✓' if 0.5 < pm/tm < 2.0 else '✗'
    print(f'  {name}: train_median={tm:.2f}  pred_median={pm:.2f}  ratio={pm/tm:.2f} {flag}')

CV RMSE в исходном пространстве (OOF):
  IC50   : 351.6768
  CC50   : 508.0314
  SI     : 783.4191
  Среднее: 547.7091

Калибровка (сравнение медиан):
  IC50: train_median=44.07  pred_median=47.50  ratio=1.08 ✓
  CC50: train_median=376.58  pred_median=300.32  ratio=0.80 ✓
  SI: train_median=4.00  pred_median=4.94  ratio=1.24 ✓


## 11. Генерация submission

In [11]:
import os
os.makedirs('../submissions', exist_ok=True)

submission = pd.DataFrame({
    'index': test['index'].values,
    'IC50':  pred_ic50,
    'CC50':  pred_cc50,
    'SI':    pred_si,
})

submission.to_csv('../submissions/submission.csv', index=False)
print('Submission сохранён в submissions/submission.csv')
print(submission.describe().round(3))
print()
print(submission.head(10).to_string())

Submission сохранён в submissions/submission.csv
         index     IC50      CC50       SI
count  250.000  250.000   250.000  250.000
mean   124.500  104.630   441.825   13.547
std     72.313  139.379   418.682   37.904
min      0.000    0.173    16.008    1.288
25%     62.250   24.502   157.853    3.124
50%    124.500   47.504   300.316    4.943
75%    186.750  114.514   615.328    8.810
max    249.000  818.205  2588.950  312.046

   index        IC50         CC50         SI
0      0   64.331825   209.526773   3.865230
1      1   95.161400   310.350541   3.361017
2      2   39.384104   242.873167   5.905021
3      3   93.081566   246.220726   3.130325
4      4  120.124384   255.161116   2.257813
5      5  102.925342   251.354680   2.491886
6      6   39.384104   242.873167   5.905021
7      7   17.806089    76.699041   4.407891
8      8   52.039486  1514.718344  25.088227
9      9   19.714741   354.125328  21.953919
